# Tome C3, Chapter 9: NP-Completeness (Python companion)

Exercises first, solutions at the end. We write a small SAT solver, then use it as the universal engine the chapter promises: we turn a Turing machine into a formula exactly as in the proof of the Cook–Levin theorem and read certificates off its solutions; we implement the reductions of Section 9.3 and check them against brute force; and we watch random 3-SAT become hard near a famous threshold. The last problems are about numbers: why subset sum is easy when its numbers are small, and how the reduction of Theorem 9.10 makes them huge.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random, time
import matplotlib.pyplot as plt
random.seed(9)

## Exercises

**P9.1 (★) A SAT solver.** Write a DPLL solver: unit propagation, then branch on a literal. Formulas are lists of clauses, clauses lists of nonzero integers ($-3$ is $\neg x_3$). Test it against brute force on random formulas, and make it count models as well.

**P9.2 (★★) The Cook–Levin formula, built.** Take a one-tape verifier which, on $x\#y$, accepts when $y$ has an odd number of $1$s. Build the formula $\varphi_x$ of Theorem 9.4 with its variables $X_{t,i,s}$, $H_{t,i}$, $S_{t,q}$ and its five groups of clauses. Check that its models correspond one to one to the accepted certificates, read a certificate off a model, and measure how the size of $\varphi_x$ grows with $T$.

**P9.3 (★) Checking is easy, finding is hard.** Write verifiers for 3-colouring and subset sum, and compare the time to check one certificate with the time to find one by trying them all, as the instance grows.

**P9.4 (★★) Reductions, checked.** Implement the reductions of 3-SAT to independent set (Theorem 9.6), 3-colouring (Theorem 9.7), subset sum (Theorem 9.10) and 0-1 integer programming (Theorem 9.12). On random small formulas, check by brute force that the formula is satisfiable exactly when the instance is a yes-instance, and translate solutions back into assignments.

**P9.5 (★★) A Hamiltonian cycle that knows an assignment.** Build the directed graph of Theorem 9.8 for a small formula, find a Hamiltonian cycle by backtracking, and decode the truth values from the directions in which the rows are traversed. Check that the clause vertices are visited by detours inside a single pair, as the proof says they must be.

**P9.6 (★) The colouring gadget.** Enumerate the proper $3$-colourings of the clause gadget of Figure 9.3 with the palette, for each colouring of the three literals, and check that the output $o'$ can be coloured exactly when some literal is true.

**P9.7 (★) Search from decision.** Using only a yes/no satisfiability oracle, find a satisfying assignment with $n$ questions (Proposition 9.17), and find a largest clique with at most $2n$ questions (Exercise 9.2).

**P9.8 (★★) Random 3-SAT and pseudo-polynomial time.** Plot, for random 3-SAT formulas with $n$ variables and $\alpha n$ clauses, the probability of satisfiability and the work of your solver as functions of $\alpha$. Then measure the dynamic programme for subset sum on the instances produced by the reduction of Theorem 9.10, and explain the numbers you see.

## Solutions

### P9.1

In [ ]:
def simplify(clauses, lit):
    # make lit true: drop satisfied clauses, remove the opposite literal
    out = []
    for c in clauses:
        if lit in c:
            continue
        if -lit in c:
            c = [l for l in c if l != -lit]
            if not c:
                return None          # empty clause: conflict
        out.append(c)
    return out

def unit_propagate(clauses, assignment, stats):
    while True:
        unit = next((c[0] for c in clauses if len(c) == 1), None)
        if unit is None:
            return clauses
        assignment[abs(unit)] = unit > 0
        stats['props'] += 1
        clauses = simplify(clauses, unit)
        if clauses is None:
            return None

def choose(clauses):
    count = {}
    for c in clauses:
        w = 2.0 ** -len(c)               # literals in short clauses count more
        for l in c:
            count[l] = count.get(l, 0) + w
    return max(count, key=count.get)

def dpll(clauses, assignment=None, stats=None):
    if assignment is None:
        assignment = {}
    if stats is None:
        stats = {'nodes': 0, 'props': 0}
    stats['nodes'] += 1
    clauses = unit_propagate([list(c) for c in clauses], assignment, stats)
    if clauses is None:
        return None
    if not clauses:
        return assignment
    lit = choose(clauses)
    for l in (lit, -lit):
        sub = simplify(clauses, l)
        if sub is not None:
            a = dict(assignment); a[abs(l)] = l > 0
            res = dpll(sub, a, stats)
            if res is not None:
                return res
    return None

def count_models(clauses, variables):
    # number of assignments of `variables` satisfying all clauses
    if clauses is None:
        return 0
    if not clauses:
        return 2 ** len(variables)
    assignment = {}
    stats = {'nodes': 0, 'props': 0}
    cl = unit_propagate([list(c) for c in clauses], assignment, stats)
    if cl is None:
        return 0
    rest = variables - set(assignment)
    if not cl:
        return 2 ** len(rest)
    v = abs(cl[0][0])
    rest = rest - {v}
    return count_models(simplify(cl, v), rest) + count_models(simplify(cl, -v), rest)

def satisfies(clauses, a):
    return all(any(a.get(abs(l), False) == (l > 0) for l in c) for c in clauses)

def random_ksat(n, m, k, rng):
    return [[v * rng.choice((1, -1)) for v in rng.sample(range(1, n + 1), k)] for _ in range(m)]

def brute(clauses, n):
    return sum(satisfies(clauses, dict(zip(range(1, n + 1), bits)))
               for bits in itertools.product((False, True), repeat=n))

rng = random.Random(1)
for trial in range(200):
    n = rng.randint(3, 10); m = rng.randint(1, 6 * n)
    F = random_ksat(n, m, 3, rng)
    a = dpll(F)
    nb = brute(F, n)
    assert (a is not None) == (nb > 0)
    if a is not None:
        assert satisfies(F, a)
    assert count_models(F, set(range(1, n + 1))) == nb
print("DPLL and model counting agree with brute force on 200 random formulas")

Unit propagation is the engine: a clause with one literal left forces that literal, which may shorten other clauses to one literal, and so on. Branching is needed only when propagation stops. The branching rule (prefer literals of short clauses) is a mild version of what real solvers do; modern solvers add *clause learning*, which records the reason for each conflict as a new clause, and handle formulas with millions of variables from hardware verification. None of this contradicts NP-completeness, which is about the worst case.

### P9.2

In [ ]:
# A one-tape verifier: on x#y it accepts iff y has an odd number of 1s.
BLANK = '_'
GAMMA = ['0', '1', '#', BLANK]
STATES = ['q0', 'even', 'odd', 'acc', 'rej']
HALT = {'acc', 'rej'}
delta = {('q0', '0'): ('q0', '0', 1), ('q0', '1'): ('q0', '1', 1), ('q0', '#'): ('even', '#', 1),
         ('q0', BLANK): ('rej', BLANK, 1),
         ('even', '0'): ('even', '0', 1), ('even', '1'): ('odd', '1', 1), ('even', BLANK): ('rej', BLANK, 1),
         ('odd', '0'): ('odd', '0', 1), ('odd', '1'): ('even', '1', 1), ('odd', BLANK): ('acc', BLANK, 1),
         ('even', '#'): ('rej', '#', 1), ('odd', '#'): ('rej', '#', 1)}

def Delta(q, s):
    return (q, s, 0) if q in HALT else delta[(q, s)]

def run(word, T):
    tape = {i: ch for i, ch in enumerate(word)}
    q, h = 'q0', 0
    for _ in range(T):
        q, tape[h], D = Delta(q, tape.get(h, BLANK))
        h += D
    return q

class Vars:
    def __init__(self):
        self.ids, self.names = {}, {}
    def __call__(self, *key):
        if key not in self.ids:
            self.ids[key] = len(self.ids) + 1
            self.names[self.ids[key]] = key
        return self.ids[key]

def exactly_one(lits):
    return [list(lits)] + [[-a, -b] for a, b in itertools.combinations(lits, 2)]

def cook_levin(x, p, T):
    V = Vars()
    X = lambda t, i, s: V('X', t, i, s)
    H = lambda t, i: V('H', t, i)
    S = lambda t, q: V('S', t, q)
    cells = range(-T, T + 1)
    F = []
    for t in range(T + 1):                                    # 1. exactly one
        for i in cells:
            F += exactly_one([X(t, i, s) for s in GAMMA])
        F += exactly_one([H(t, i) for i in cells])
        F += exactly_one([S(t, q) for q in STATES])
    n = len(x)                                                # 2. start
    F += [[S(0, 'q0')], [H(0, 0)]]
    for i in cells:
        if 0 <= i < n:
            F.append([X(0, i, x[i])])
        elif i == n:
            F.append([X(0, i, '#')])
        elif n < i <= n + p:
            F += [[-X(0, i, s)] for s in GAMMA if s not in '01']
        else:
            F.append([X(0, i, BLANK)])
    for t in range(T):                                        # 3. moves and 4. frame
        for i in cells:
            for q in STATES:
                for s in GAMMA:
                    q2, s2, D = Delta(q, s)
                    pre = [-H(t, i), -S(t, q), -X(t, i, s)]
                    if -T <= i + D <= T:
                        F += [pre + [S(t + 1, q2)], pre + [X(t + 1, i, s2)], pre + [H(t + 1, i + D)]]
                    else:
                        F.append(pre)
            for s in GAMMA:
                F.append([H(t, i), -X(t, i, s), X(t + 1, i, s)])
    F.append([S(T, 'acc')])                                   # 5. accept
    return F, V

x, p = '011', 3
T = len(x) + p + 2
F, V = cook_levin(x, p, T)
allvars = set(V.ids.values())
print(f"x = {x}, certificates of length {p}, T = {T}: {len(allvars)} variables, {len(F)} clauses")

accepted = [''.join(y) for y in itertools.product('01', repeat=p) if run(x + '#' + ''.join(y), T) == 'acc']
models = count_models(F, allvars)
print("certificates accepted by the verifier:", accepted)
print("models of the formula:", models)

a = dpll(F)
y = ''.join(next(s for s in '01' if a.get(V.ids[('X', 0, len(x) + 1 + k, s)], False)) for k in range(p))
print("certificate read off a model:", y, "-> verifier says", run(x + '#' + y, T))

sizes = []
for pp in [2, 4, 8, 12, 16, 20]:
    TT = len(x) + pp + 2
    FF, VV = cook_levin(x, pp, TT)
    sizes.append((TT, len(VV.ids), len(FF), sum(len(c) for c in FF)))
for TT, nv, nc, nl in sizes:
    print(f"T = {TT:2d}: {nv:6d} variables, {nc:7d} clauses, {nl:8d} literals")
plt.figure(figsize=(6, 3.5))
Ts = [s[0] for s in sizes]
plt.loglog(Ts, [s[2] for s in sizes], 'o-', label='clauses')
plt.loglog(Ts, [s[1] for s in sizes], 's-', label='variables')
plt.loglog(Ts, [30 * t ** 3 for t in Ts], 'k--', lw=0.8, label=r'$\propto T^3$')
plt.loglog(Ts, [20 * t ** 2 for t in Ts], 'k:', lw=0.8, label=r'$\propto T^2$')
plt.xlabel('T'); plt.legend(); plt.tight_layout(); plt.show()

The formula has exactly as many models as there are accepted certificates (here the four words of length $3$ with an odd number of $1$s): once the certificate cells of the first row are chosen, unit propagation fills in the whole table, row after row, because the verifier is deterministic. That is the proof of Theorem 9.4 running. The solver never "understood" parity; it found a certificate by propagating the clauses of a table.

The sizes follow the count in the proof: $O(T^2)$ variables, and $O(T^3)$ clauses, the cubic term coming from the clauses "at most one head position" at each time; with a binary encoding of the head position it would drop to $O(T^2\log T)$. For a verifier running in time $n^k$, the formula has size polynomial in $n$: that is all the theorem needs.

### P9.3

In [ ]:
def verify_colouring(edges, col):
    return all(col[u] != col[v] for u, v in edges)

def find_colouring(n, edges):
    for col in itertools.product(range(3), repeat=n):
        if verify_colouring(edges, col):
            return col
    return None

def verify_subset(a, t, bits):
    return sum(x for x, b in zip(a, bits) if b) == t

def find_subset(a, t):
    for bits in itertools.product((0, 1), repeat=len(a)):
        if verify_subset(a, t, bits):
            return bits
    return None

rng = random.Random(3)
print(" n | check colouring | find colouring (no-instance) | check subset | find subset (no-instance)")
rows = []
for n in [6, 8, 10, 12, 14]:
    # an odd wheel plus a disjoint path: not 3-colourable, so the search tries everything
    edges = [(0, i) for i in range(1, 6)] + [(i, i % 5 + 1) for i in range(1, 6)] + [(i, i + 1) for i in range(6, n - 1)]
    col = tuple(rng.randrange(3) for _ in range(n))
    t0 = time.perf_counter(); verify_colouring(edges, col); tc = time.perf_counter() - t0
    t0 = time.perf_counter(); res = find_colouring(n, edges); tf = time.perf_counter() - t0
    assert res is None
    a = [2 * rng.randint(1, 10 ** 6) for _ in range(n + 4)]
    t = 2 * sum(a) // 4 + 1                    # odd target, all numbers even: no solution
    bits = [rng.randrange(2) for _ in a]
    t0 = time.perf_counter(); verify_subset(a, t, bits); ts = time.perf_counter() - t0
    t0 = time.perf_counter(); find_subset(a, t); tfs = time.perf_counter() - t0
    rows.append((n, tf, tfs))
    print(f"{n:2d} | {tc * 1e6:12.1f} µs | {tf:24.3f} s   | {ts * 1e6:9.1f} µs | {tfs:18.3f} s")

Checking a certificate takes microseconds and grows linearly; the exhaustive search grows by a factor $9$ for every two vertices (three colours each) and by a factor $4$ for every two numbers. (A wheel with five spokes needs four colours, so these are no-instances and the search has to try everything.) Whether the gap is real, whether some cleverer search avoids it for every NP problem, is the question $\mathsf P=\mathsf{NP}$. For these two problems we know cleverer algorithms, in time about $1.33^n$ for 3-colouring and $2^{N/2}$ for subset sum (meet in the middle), but all known algorithms remain exponential.

### P9.4

In [ ]:
def to_independent_set(F):
    V = [(j, l) for j, c in enumerate(F) for l in c]
    E = set()
    for (j1, l1), (j2, l2) in itertools.combinations(V, 2):
        if j1 == j2 or l1 == -l2:
            E.add(((j1, l1), (j2, l2)))
    return V, E, len(F)

def has_independent_set(V, E, k):
    adj = {v: set() for v in V}
    for u, v in E:
        adj[u].add(v); adj[v].add(u)
    def rec(cands, size):
        if size >= k:
            return []
        for idx, v in enumerate(cands):
            rest = [w for w in cands[idx + 1:] if w not in adj[v]]
            if size + 1 + len(rest) < k:
                continue
            r = rec(rest, size + 1)
            if r is not None:
                return [v] + r
        return None
    return rec(list(V), 0)

def to_3col(F, n):
    G = {'T': {'F', 'B'}, 'F': {'T', 'B'}, 'B': {'T', 'F'}}
    def edge(u, v):
        G.setdefault(u, set()).add(v); G.setdefault(v, set()).add(u)
    lit = lambda l: ('x', l)
    for i in range(1, n + 1):
        edge(lit(i), lit(-i)); edge(lit(i), 'B'); edge(lit(-i), 'B')
    for j, c in enumerate(F):
        a, b, cc = (list(c) + [c[0], c[0]])[:3]
        p, q, o, p2, q2, o2 = [(j, s) for s in ('p', 'q', 'o', "p'", "q'", "o'")]
        for u, v in [(lit(a), p), (lit(b), q), (p, q), (q, o), (o, p), (o, p2), (lit(cc), q2), (p2, q2), (q2, o2), (o2, p2), (o2, 'F'), (o2, 'B')]:
            edge(u, v)
    return G

def three_colour(G):
    order = sorted(G, key=lambda v: -len(G[v]))
    order.remove('T'); order.remove('F'); order.remove('B')
    col = {'T': 0, 'F': 1, 'B': 2}
    def rec(k):
        if k == len(order):
            return True
        v = order[k]
        for c in range(3):
            if all(col.get(u) != c for u in G[v]):
                col[v] = c
                if rec(k + 1):
                    return True
                del col[v]
        return False
    return dict(col) if rec(0) else None

def to_subset_sum(F, n):
    m = len(F)
    def number(var_col, clause_cols):
        digits = [0] * (n + m)
        if var_col is not None:
            digits[var_col] = 1
        for j in clause_cols:
            digits[n + j] += 1
        return int(''.join(map(str, digits)))
    nums, labels = [], []
    for i in range(1, n + 1):
        nums.append(number(i - 1, [j for j, c in enumerate(F) if i in c])); labels.append(i)
        nums.append(number(i - 1, [j for j, c in enumerate(F) if -i in c])); labels.append(-i)
    for j in range(m):
        nums += [number(None, [j])] * 2; labels += [None, None]
    target = int('1' * n + '3' * m)
    return nums, labels, target

def subset_sum_solution(nums, t):
    reach = {0: ()}
    for k, a in enumerate(nums):
        new = dict(reach)
        for s, sub in reach.items():
            if s + a <= t and s + a not in new:
                new[s + a] = sub + (k,)
        reach = new
    return reach.get(t)

def to_ip(F, n):
    # rows of A x <= b, for x in {0,1}^n: a clause gives -sum(literals) <= -1
    rows = []
    for c in F:
        coef, const = [0] * n, 0
        for l in c:
            if l > 0:
                coef[l - 1] += 1
            else:
                coef[-l - 1] -= 1; const += 1
        rows.append(([-v for v in coef], const - 1))
    return rows

def ip_feasible(rows, n):
    for x in itertools.product((0, 1), repeat=n):
        if all(sum(a * xi for a, xi in zip(coef, x)) <= b for coef, b in rows):
            return x
    return None

rng = random.Random(4)
stats = {'is': 0, 'col': 0, 'ss': 0, 'ip': 0, 'sat': 0}
for trial in range(150):
    n = rng.randint(2, 5); m = rng.randint(2, 7)
    F = [[v * rng.choice((1, -1)) for v in rng.sample(range(1, min(n, 3) + 1) if n < 3 else range(1, n + 1), rng.choice((1, 2, 3, 3, 3)) if n >= 3 else rng.choice((1, 2)))] for _ in range(m)]
    sat = brute(F, n) > 0
    stats['sat'] += sat
    V, E, k = to_independent_set(F)
    S = has_independent_set(V, E, k)
    assert (S is not None) == sat
    if S:
        a = {abs(l): l > 0 for _, l in S}
        assert satisfies(F, a)
    if trial < 60:
        col = three_colour(to_3col(F, n))
        assert (col is not None) == sat
        if col:
            assert satisfies(F, {i: col[('x', i)] == 0 for i in range(1, n + 1)})
    nums, labels, t = to_subset_sum(F, n)
    sol = subset_sum_solution(nums, t)
    assert (sol is not None) == sat
    if sol:
        a = {abs(labels[k]): labels[k] > 0 for k in sol if labels[k] is not None}
        assert satisfies(F, a)
    assert (ip_feasible(to_ip(F, n), n) is not None) == sat
print(f"150 random formulas ({stats['sat']} satisfiable): all four reductions agree with brute force,")
print("and every solution of a reduced instance decodes to a satisfying assignment.")
F = [[1, -2, 3], [-1, 2], [2, 3, -4]]
nums, labels, t = to_subset_sum(F, 4)
print("\nsubset sum instance for", F)
for x, l in zip(nums, labels):
    print(f"  {x:0{len(str(t))}d}  {'u' if l and l > 0 else 'v' if l else 'slack'}{abs(l) if l else ''}")
print("  target", t)

Every reduction works in both directions, and every solution of the reduced instance can be translated back: the independent set names one true literal per clause, the colouring gives the colour of $T$ to the true literals, the subset says which of $u_i$, $v_i$ was chosen. That is typical: the reductions of Section 9.3 are *parsimonious enough* to carry solutions, not just answers. (For the subset sum instance, look at the columns: no column can carry, since its digits add up to at most $5$.)

The independent set search and the colouring search are themselves exponential; the point is only that their answers match.

### P9.5

In [ ]:
def ham_graph(F, n):
    m = len(F)
    L = 3 * m + 3
    out = {}
    def arc(u, v):
        out.setdefault(u, []).append(v); out.setdefault(v, [])
    for i in range(1, n + 1):
        w = lambda p: ('w', i, p)
        for p in range(L - 1):
            arc(w(p), w(p + 1)); arc(w(p + 1), w(p))
        arc(('s', i), w(0)); arc(('s', i), w(L - 1)); arc(w(0), ('t', i)); arc(w(L - 1), ('t', i))
        arc(('t', i), ('s', i % n + 1))
        for j, c in enumerate(F, start=1):
            if i in c:
                arc(w(3 * j - 1), ('c', j)); arc(('c', j), w(3 * j))
            if -i in c:
                arc(w(3 * j), ('c', j)); arc(('c', j), w(3 * j - 1))
    return out

def ham_cycle(out, start):
    N = len(out)
    pred = {v: [] for v in out}
    for u in out:
        for v in out[u]:
            pred[v].append(u)
    path, seen = [start], {start}
    def ok_degrees():
        # every unvisited vertex still needs an unvisited (or current/start) predecessor and successor
        for v in out:
            if v in seen:
                continue
            if not any(u not in seen or u == path[-1] for u in pred[v]):
                return False
            if not any(u not in seen or u == start for u in out[v]):
                return False
        return True
    def rec():
        u = path[-1]
        if len(path) == N:
            return start in out[u]
        for v in out[u]:
            if v not in seen:
                path.append(v); seen.add(v)
                if ok_degrees() and rec():
                    return True
                path.pop(); seen.discard(v)
        return False
    return list(path) if rec() else None

F = [[1, -2, 3], [-1, 2, -3], [1, 2]]
n = 3
G = ham_graph(F, n)
print(f"formula {F}: graph with {len(G)} vertices and {sum(map(len, G.values()))} arcs")
t0 = time.perf_counter()
H = ham_cycle(G, ('s', 1))
print(f"Hamiltonian cycle found in {time.perf_counter() - t0:.2f} s")
pos = {v: k for k, v in enumerate(H)}
assignment = {}
for i in range(1, n + 1):
    nxt = H[(pos[('s', i)] + 1) % len(H)]
    assignment[i] = nxt == ('w', i, 0)          # left to right means true
print("decoded assignment:", assignment, "satisfies F:", satisfies(F, assignment))
for j in range(1, len(F) + 1):
    k = pos[('c', j)]
    a, b = H[k - 1], H[(k + 1) % len(H)]
    print(f"clause c{j}: detour {a} -> c{j} -> {b}, same row and pair: {a[1] == b[1] and abs(a[2] - b[2]) == 1}")

The cycle runs through each row in one direction, and the direction is the truth value; each clause vertex is visited by a detour between the two vertices of one pair, in the direction allowed by a true literal. The backtracking search needs the pruning rule (every unvisited vertex must still be enterable and leavable) to finish quickly: without it, it explores dead ends that the separators of the construction make obvious to a human but not to a naive program.

### P9.6

In [ ]:
def gadget_outputs(a, b, c):
    # colours: 0 = T, 1 = F, 2 = B; literals a, b, c are 0 or 1
    names = ['p', 'q', 'o', "p'", "q'", "o'"]
    E = [('a', 'p'), ('b', 'q'), ('p', 'q'), ('q', 'o'), ('o', 'p'), ('o', "p'"), ('c', "q'"),
         ("p'", "q'"), ("q'", "o'"), ("o'", "p'"), ("o'", 'F'), ("o'", 'B')]
    fixed = {'a': a, 'b': b, 'c': c, 'F': 1, 'B': 2}
    possible_o, possible_o2 = set(), set()
    for cols in itertools.product(range(3), repeat=6):
        col = dict(fixed, **dict(zip(names, cols)))
        if all(col[u] != col[v] for u, v in E):
            possible_o.add(col['o']); possible_o2.add(col["o'"])
    return possible_o, possible_o2

name = {0: 'T', 1: 'F', 2: 'B'}
for a, b, c in itertools.product((0, 1), repeat=3):
    po, po2 = gadget_outputs(a, b, c)
    print(f"a={name[a]} b={name[b]} c={name[c]}: o can be {sorted(name[x] for x in po)}, "
          f"gadget colourable: {bool(po2)}")

The table is the proof of Theorem 9.7 in miniature. When $a$ and $b$ are both false, $o$ can only be false; otherwise it can be true (when both are true, it must be). And the whole gadget, with $o'$ tied to $F$ and $B$, can be coloured in every case except one: all three literals false.

### P9.7

In [ ]:
calls = {'n': 0}
def sat_oracle(F):
    calls['n'] += 1
    return dpll(F) is not None

def find_assignment(F, n):
    a = {}
    for i in range(1, n + 1):
        G = simplify(F, i)
        if G is not None and sat_oracle(G):
            a[i], F = True, G
        else:
            a[i], F = False, simplify(F, -i)
    return a

rng = random.Random(7)
n = 40
while True:
    F = random_ksat(n, int(3.8 * n), 3, rng)
    if dpll(F) is not None:
        break
calls['n'] = 0
a = find_assignment(F, n)
print(f"n = {n}: assignment found with {calls['n']} oracle calls, satisfies F: {satisfies(F, a)}")

def clique_oracle(adj, vertices, k):
    calls['n'] += 1
    vs = sorted(vertices)
    def rec(cands, size):
        if size == k:
            return True
        for idx, v in enumerate(cands):
            if size + len(cands) - idx < k:
                return False
            if rec([w for w in cands[idx + 1:] if w in adj[v]], size + 1):
                return True
        return False
    return rec(vs, 0)

def largest_clique(adj):
    V = set(adj)
    k = 0
    while clique_oracle(adj, V, k + 1):
        k += 1
    for v in sorted(adj):
        if clique_oracle(adj, V - {v}, k):
            V = V - {v}
    return V

n = 30
adj = {v: set() for v in range(n)}
for u, v in itertools.combinations(range(n), 2):
    if rng.random() < 0.5:
        adj[u].add(v); adj[v].add(u)
calls['n'] = 0
K = largest_clique(adj)
print(f"G(30, 1/2): largest clique {sorted(K)} of size {len(K)}, "
      f"is a clique: {all(v in adj[u] for u, v in itertools.combinations(K, 2))}, oracle calls: {calls['n']}")

With at most $n$ questions to a satisfiability oracle we fix the variables one at a time, always keeping a satisfiable formula (here fewer than $40$, since no question is needed when setting $x_i$ true creates an empty clause); with at most $n+\omega+1$ questions to a clique oracle we find a largest clique. So a polynomial algorithm for the *decision* problems would give polynomial algorithms for the *search* problems: decision is the heart of the matter, which is why complexity theory studies languages.

### P9.8

In [ ]:
rng = random.Random(8)
alphas = [2.0, 3.0, 3.5, 4.0, 4.27, 4.5, 5.0, 6.0, 7.0]
plt.figure(figsize=(9, 3.5))
for n in [20, 40]:
    probs, work = [], []
    for alpha in alphas:
        trials = 30 if n == 20 else 12
        sat = 0; nodes = []
        for _ in range(trials):
            st = {'nodes': 0, 'props': 0}
            F = random_ksat(n, int(alpha * n), 3, rng)
            sat += dpll(F, stats=st) is not None
            nodes.append(st['nodes'])
        probs.append(sat / trials); work.append(sorted(nodes)[len(nodes) // 2])
    plt.subplot(1, 2, 1); plt.plot(alphas, probs, 'o-', label=f'n = {n}')
    plt.subplot(1, 2, 2); plt.semilogy(alphas, work, 'o-', label=f'n = {n}')
    print(f"n = {n}: median DPLL nodes by alpha:", dict(zip(alphas, work)))
plt.subplot(1, 2, 1); plt.axvline(4.27, color='k', ls='--', lw=0.8); plt.xlabel(r'$\alpha$'); plt.ylabel('P(satisfiable)'); plt.legend()
plt.subplot(1, 2, 2); plt.axvline(4.27, color='k', ls='--', lw=0.8); plt.xlabel(r'$\alpha$'); plt.ylabel('median search nodes'); plt.legend()
plt.tight_layout(); plt.show()

# pseudo-polynomial: the subset sum DP on reduced instances
def dp_states(nums, t):
    reach = {0}
    for a in nums:
        reach |= {s + a for s in reach if s + a <= t}
    return len(reach)

print("\nsubset sum DP on instances produced by Theorem 9.10:")
for n in [2, 3, 4, 5]:
    m = n + 1
    F = random_ksat(n, m, 3, rng) if n >= 3 else random_ksat(n, m, 2, rng)
    nums, labels, t = to_subset_sum(F, n)
    t0 = time.perf_counter()
    st = dp_states(nums, t)
    print(f"  n = {n}, m = {m}: target {t} ({t.bit_length()} bits), "
          f"DP keeps {st} sums, {time.perf_counter() - t0:.3f} s")

Random 3-SAT shows a threshold: below about $4.27$ clauses per variable formulas are almost always satisfiable, above it almost never, and the search is hardest near the threshold, where the solver must work to find a rare solution or to prove there is none. (The threshold is a conjecture supported by statistical physics; Jian Ding, Allan Sly and Nike Sun proved in 2015 that a sharp threshold exists for $k$-SAT with $k$ large, and its location for $k=3$ is still open.) Far from it, easy instances dominate. NP-hardness is about the worst case, and random instances are a different world.

The dynamic programme for subset sum keeps one entry per reachable sum below $t$. On the reduced instances $t$ has $n+m$ decimal digits, so the bound $O(Nt)$ is about $10^{n+m}$; the set actually kept is much smaller than $t$, but it grows by a factor of about $11$ each time $n$ and $m$ both increase by one: exponential in the size of the formula, as it must be unless $\mathsf P=\mathsf{NP}$. In general the running time $O(Nt)$ is exponential in the length of $t$: subset sum is NP-complete, but only *weakly*, and with numbers bounded by a polynomial (in unary) it would be in $\mathsf P$, as in Exercise 9.9.